In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
import sys
import asyncio
""" Docstring for the module.
This module provides a workaround for known issues when running MCP servers in
Jupyter notebooks on Windows. It ensures that the appropriate event loop policy is 
set and redirects stderr to avoid fileno() errors.
"""
# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
""" Docstring for the module.
This module initializes a MultiServerMCPClient with a configuration for a local MCP server."""

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [ ]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt=prompt
)

In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [ ]:
from pprint import pprint

pprint(response["messages"][-1].content)

('Here’s a concise overview of the langchain-mcp-adapters library and how it '
 'fits into LangChain/LangGraph/LangSmith ecosystems.\n'
 '\n'
 'What it is\n'
 '- A bridge between MCP (Model Context Protocol) tool servers and '
 'LangChain/LangGraph applications.\n'
 '- Available for both Python (LangChain) and JavaScript/Node ecosystems '
 '(LangChain.js, LangGraph.js).\n'
 '\n'
 'What MCP adapters do\n'
 '- Convert MCP tools, prompts, and resources into LangChain-compatible '
 'primitives.\n'
 '- Allow agents to call tools from one or more MCP servers as if they were '
 'native LangChain tools.\n'
 '- Enable interaction with multiple MCP servers seamlessly (multi-server '
 'tooling).\n'
 '\n'
 'Key components\n'
 '- Tools adapter: Converts MCP tools into LangChain-compatible tools that can '
 'be used by agents.\n'
 '- Prompts adapter: Transforms MCP prompts into LangChain messages/prompts.\n'
 '- Resources adapter: Converts MCP resources into LangChain Blob objects '
 '(text and bina

## Online MCP

In [ ]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [ ]:
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
)

In [ ]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response["messages"][-1].content)

("It's 7:54 AM on Thursday, June 4, 2026 in New York (Eastern Daylight Time, "
 'UTC-4). Want me to convert this to another time zone?')
